# EP10. 멀티 에이전트 시스템 심화

## 학습 목표

1. **LangGraph Supervisor 패턴**을 구현하여 에이전트 간 위임 로직을 제어한다
2. **LangGraph Swarm 패턴**으로 handoff 기반 자율 협업 파이프라인을 구성한다
3. **deepagents** (`create_deep_agent` + `subagents`)로 3-에이전트 코드 리뷰 파이프라인을 빠르게 구축한다
4. **Langfuse**로 멀티에이전트 실행 흐름을 트레이스 트리로 시각화한다

## AI 가이드

> 막히는 부분은 Claude에게 물어보세요.
> 예시 프롬프트:
> - "LangGraph StateGraph에서 Supervisor 노드가 Worker를 선택하는 로직을 설명해줘"
> - "create_deep_agent에서 subagents 파라미터 구조를 알려줘"
> - "LangGraph Command와 goto를 사용해서 조건부 분기를 만드는 방법을 알려줘"
> - "Langfuse CallbackHandler를 LangGraph invoke에 붙이는 방법을 알려줘"

**사전 요구사항**: EP06 (Langfuse 기초)

**예상 소요 시간**: 90~120분

**필요한 API 키**:
- `ANTHROPIC_API_KEY`
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`

**전체 아키텍처**:
```
사용자 요청
    ↓
[기획자 에이전트] → 요구사항 분석 + 계획 수립
    ↓ subagents 위임
[개발자 에이전트] → Python 코드 작성
[검수자 에이전트] → 버그 탐지 + 리뷰
    ↓
[기획자 에이전트] → 최종 결과 통합
    ↓
Langfuse 트레이스 트리 시각화
```

## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langgraph langchain-anthropic deepagents langfuse python-dotenv

## 섹션 2. 라이브러리 로드 + Langfuse 초기화

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# 필수 API 키 확인
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다"
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "LANGFUSE_PUBLIC_KEY가 설정되지 않았습니다"
assert os.getenv("LANGFUSE_SECRET_KEY"), "LANGFUSE_SECRET_KEY가 설정되지 않았습니다"

print("API 키 로드 완료")

In [ ]:
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

# Langfuse 클라이언트 초기화
langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
)

# 연결 확인
langfuse.auth_check()
print("Langfuse 연결 성공")

## 섹션 3. 멀티에이전트 개념 이해: 단일 에이전트 한계 실험

복잡한 태스크(코드 작성 + 리뷰)를 단일 에이전트에게 주면 어떤 결과가 나오는지 확인합니다.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

# 단일 에이전트 (범용 시스템 프롬프트)
single_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

SINGLE_TASK = """피보나치 수열을 계산하는 Python 함수를 작성하고,
그 코드에 대한 코드 리뷰(버그, 성능, 가독성)도 함께 제공해주세요."""

# 단일 에이전트 실행
single_response = single_llm.invoke([HumanMessage(content=SINGLE_TASK)])
single_output = single_response.content

print("=== 단일 에이전트 결과 ===")
print(single_output[:1000])
print(f"\n총 글자 수: {len(single_output)}")

In [ ]:
# 단일 에이전트의 한계 분석
print("단일 에이전트 문제점:")
print("1. 코드 작성과 리뷰를 동시에 하므로 자기 코드를 스스로 리뷰 → 객관성 부족")
print("2. 시스템 프롬프트가 범용적이어서 전문성이 낮음")
print("3. 복잡한 태스크일수록 초기 요구사항 분석이 피상적")
print()
print("→ 역할 분리된 멀티 에이전트 시스템이 필요한 이유")

## 섹션 4. LangGraph Supervisor 패턴 구현

Supervisor 에이전트가 태스크를 분석하고 적절한 Worker에게 위임합니다.

In [ ]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 에이전트 간 공유되는 State 정의
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # 전체 대화 히스토리
    next: str                                 # 다음 실행할 에이전트
    code: str                                 # 작성된 코드 저장
    review: str                               # 리뷰 결과 저장

# LLM 인스턴스
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

WORKERS = ["code_writer", "code_reviewer", "FINISH"]

print("State 및 LLM 정의 완료")

In [ ]:
import json

def supervisor_node(state: AgentState) -> Command[Literal["code_writer", "code_reviewer", "__end__"]]:
    """Supervisor: 어떤 Worker에게 태스크를 위임할지 결정"""
    
    system_prompt = """당신은 소프트웨어 팀 매니저입니다.
현재 상황을 분석하고 다음에 실행할 에이전트를 결정하세요.

사용 가능한 에이전트:
- code_writer: 코드 작성이 필요할 때
- code_reviewer: 코드가 작성되었고 리뷰가 필요할 때
- FINISH: 모든 작업이 완료되었을 때

반드시 다음 JSON 형식으로만 응답하세요:
{"next": "<에이전트 이름>", "reason": "<선택 이유>"}"""
    
    messages = [
        SystemMessage(content=system_prompt),
    ] + state["messages"]
    
    # 현재 상태 컨텍스트 추가
    context = f"현재 상태: 코드={'있음' if state.get('code') else '없음'}, 리뷰={'있음' if state.get('review') else '없음'}"
    messages.append(HumanMessage(content=context))
    
    response = llm.invoke(messages)
    
    # JSON 파싱
    try:
        decision = json.loads(response.content)
        next_agent = decision.get("next", "FINISH")
        reason = decision.get("reason", "")
    except (json.JSONDecodeError, AttributeError):
        # 파싱 실패 시 현재 상태로 결정
        if not state.get("code"):
            next_agent = "code_writer"
        elif not state.get("review"):
            next_agent = "code_reviewer"
        else:
            next_agent = "FINISH"
        reason = "자동 결정"
    
    print(f"[Supervisor] 다음: {next_agent} / 이유: {reason}")
    
    # Command로 다음 노드 지정
    goto = "__end__" if next_agent == "FINISH" else next_agent
    return Command(
        goto=goto,
        update={"next": next_agent}
    )

print("Supervisor 노드 정의 완료")

In [ ]:
def code_writer_node(state: AgentState) -> Command[Literal["supervisor"]]:
    """Worker 1: Python 코드 작성 전문 에이전트"""
    
    system_prompt = """당신은 Python 개발 전문가입니다.
요청된 기능을 구현하는 깔끔하고 효율적인 Python 코드를 작성하세요.
코드만 작성하고 불필요한 설명은 최소화하세요.
반드시 함수/클래스 형태로 작성하고 docstring을 포함하세요."""
    
    # 원래 요청 메시지 추출
    user_request = ""
    for msg in state["messages"]:
        if hasattr(msg, 'content') and isinstance(msg, HumanMessage):
            user_request = msg.content
            break
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"다음 기능을 구현하는 Python 코드를 작성해주세요:\n{user_request}")
    ]
    
    response = llm.invoke(messages)
    code = response.content
    
    print(f"[개발자] 코드 작성 완료 ({len(code)} 글자)")
    
    return Command(
        goto="supervisor",
        update={
            "code": code,
            "messages": [AIMessage(content=f"[개발자] 코드 작성 완료:\n{code}")]
        }
    )


def code_reviewer_node(state: AgentState) -> Command[Literal["supervisor"]]:
    """Worker 2: 코드 리뷰 전문 에이전트"""
    
    system_prompt = """당신은 시니어 소프트웨어 엔지니어입니다.
제공된 Python 코드를 다음 관점에서 리뷰하세요:
1. 버그 및 논리 오류
2. 성능 최적화 가능성
3. 가독성 및 코드 스타일
4. 엣지 케이스 처리

각 항목에 점수(1-10)와 구체적인 개선사항을 제시하세요."""
    
    code = state.get("code", "")
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"다음 Python 코드를 리뷰해주세요:\n\n```python\n{code}\n```")
    ]
    
    response = llm.invoke(messages)
    review = response.content
    
    print(f"[검수자] 코드 리뷰 완료 ({len(review)} 글자)")
    
    return Command(
        goto="supervisor",
        update={
            "review": review,
            "messages": [AIMessage(content=f"[검수자] 리뷰 완료:\n{review}")]
        }
    )

print("Worker 노드 정의 완료")

In [ ]:
# StateGraph 구성
builder = StateGraph(AgentState)

# 노드 등록
builder.add_node("supervisor", supervisor_node)
builder.add_node("code_writer", code_writer_node)
builder.add_node("code_reviewer", code_reviewer_node)

# 시작점 설정: START → supervisor
builder.add_edge(START, "supervisor")

# 그래프 컴파일
supervisor_graph = builder.compile()

print("LangGraph Supervisor 그래프 컴파일 완료")

# 그래프 시각화 (선택사항)
try:
    from IPython.display import Image, display
    display(Image(supervisor_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("그래프 시각화 생략 (graphviz 미설치)")

In [ ]:
# Supervisor 패턴 실행
langfuse_handler = CallbackHandler(
    tags=["ep10", "supervisor-pattern", "langgraph"],
    session_id="ep10-supervisor-demo",
)

input_state = {
    "messages": [HumanMessage(content="피보나치 수열을 반환하는 Python 함수를 구현해주세요")],
    "code": "",
    "review": "",
    "next": "",
}

result = supervisor_graph.invoke(
    input_state,
    config={
        "callbacks": [langfuse_handler],
        "recursion_limit": 10,
    }
)

print("\n=== 최종 결과 ===")
print("\n--- 작성된 코드 ---")
print(result["code"][:500] if result.get("code") else "없음")
print("\n--- 리뷰 결과 ---")
print(result["review"][:500] if result.get("review") else "없음")

## 섹션 5. LangGraph Swarm 패턴 (handoff 도구 기반)

에이전트가 스스로 다음 에이전트에게 제어권을 넘기는 패턴입니다.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Handoff 도구 정의: 제어권 이전 신호
@tool
def transfer_to_developer(task_description: str) -> str:
    """개발자 에이전트에게 코드 작성 태스크를 넘깁니다.
    
    Args:
        task_description: 구현해야 할 기능의 상세 설명
    """
    return f"[Handoff to Developer] {task_description}"


@tool
def transfer_to_reviewer(code_to_review: str) -> str:
    """검수자 에이전트에게 코드 리뷰를 요청합니다.
    
    Args:
        code_to_review: 리뷰할 Python 코드
    """
    return f"[Handoff to Reviewer] 다음 코드를 리뷰해주세요:\n{code_to_review}"


# Swarm 상태 정의
class SwarmState(TypedDict):
    messages: Annotated[list, add_messages]
    current_agent: str

print("Swarm 패턴 도구 및 상태 정의 완료")

In [ ]:
# Swarm 에이전트 구성
planner_system = """당신은 기획자입니다.
사용자 요청을 분석하고 개발자에게 코드 작성을 위임하세요.
transfer_to_developer 도구를 사용하여 명확한 스펙을 전달하세요."""

developer_system = """당신은 Python 개발자입니다.
요청된 기능을 구현하는 코드를 작성하세요.
코드가 완성되면 transfer_to_reviewer 도구로 검수자에게 넘기세요."""

reviewer_system = """당신은 코드 검수자입니다.
제공된 코드를 분석하여 버그, 성능, 가독성 관점에서 리뷰를 제공하세요.
리뷰가 완료되면 최종 결과를 요약하세요."""

# 각 에이전트 생성 (ReAct 에이전트)
swarm_planner = create_react_agent(
    llm,
    tools=[transfer_to_developer],
    state_modifier=planner_system,
)

swarm_developer = create_react_agent(
    llm,
    tools=[transfer_to_reviewer],
    state_modifier=developer_system,
)

swarm_reviewer = create_react_agent(
    llm,
    tools=[],
    state_modifier=reviewer_system,
)

print("Swarm 에이전트 생성 완료")

In [ ]:
# Swarm 파이프라인 순차 실행 (간단 버전)
def run_swarm_pipeline(task: str, verbose: bool = True) -> dict:
    """기획자 → 개발자 → 검수자 순서로 Swarm 파이프라인 실행"""
    
    results = {}
    
    # 1. 기획자 실행
    if verbose:
        print("[1/3] 기획자 에이전트 실행 중...")
    planner_result = swarm_planner.invoke(
        {"messages": [HumanMessage(content=task)]}
    )
    plan_output = planner_result["messages"][-1].content
    results["plan"] = plan_output
    
    # 2. 개발자 실행
    if verbose:
        print("[2/3] 개발자 에이전트 실행 중...")
    dev_result = swarm_developer.invoke(
        {"messages": [HumanMessage(content=f"다음 태스크를 구현하세요: {task}\n\n기획자 분석: {plan_output}")]}
    )
    code_output = dev_result["messages"][-1].content
    results["code"] = code_output
    
    # 3. 검수자 실행
    if verbose:
        print("[3/3] 검수자 에이전트 실행 중...")
    review_result = swarm_reviewer.invoke(
        {"messages": [HumanMessage(content=f"다음 코드를 리뷰해주세요:\n{code_output}")]}
    )
    review_output = review_result["messages"][-1].content
    results["review"] = review_output
    
    return results


# 실행 테스트
swarm_results = run_swarm_pipeline("이진 탐색 알고리즘을 Python으로 구현해주세요")
print("\n=== Swarm 파이프라인 완료 ===")
print(f"기획 결과: {len(swarm_results['plan'])} 글자")
print(f"코드 결과: {len(swarm_results['code'])} 글자")
print(f"리뷰 결과: {len(swarm_results['review'])} 글자")

## 섹션 6. deepagents로 3-에이전트 코드 리뷰 파이프라인

`create_deep_agent`의 `subagents` 파라미터로 기획자 → 개발자 → 검수자 파이프라인을 간결하게 구성합니다.

In [ ]:
from deepagents import create_deep_agent

# 3-에이전트 코드 리뷰 파이프라인 구성
code_review_agent = create_deep_agent(
    model="anthropic:claude-haiku-4-5-20251001",
    tools=[],
    system_prompt="""당신은 소프트웨어 개발 팀 기획자입니다.
복잡한 코딩 태스크를 다음 순서로 처리하세요:
1. 요구사항을 분석하고 구현 계획을 수립합니다
2. 개발자(developer) 서브에이전트에게 코드 작성을 위임합니다
3. 검수자(reviewer) 서브에이전트에게 코드 리뷰를 위임합니다
4. 코드와 리뷰를 통합하여 최종 결과를 제공합니다

항상 서브에이전트에게 충분한 컨텍스트를 전달하세요.""",
    subagents=[
        {
            "name": "developer",
            "description": "Python 코드를 전문적으로 작성하는 개발자 에이전트. "
                           "기능 구현, 알고리즘 최적화, 클린 코드 작성을 담당합니다.",
            "model": "anthropic:claude-haiku-4-5-20251001",
            "system_prompt": """당신은 Python 개발 전문가입니다.
다음 원칙을 지키며 코드를 작성하세요:
- 명확한 함수/클래스 구조
- 타입 힌트 사용
- 상세한 docstring
- 예외 처리 포함
- 사용 예시 제공""",
            "tools": [],
        },
        {
            "name": "reviewer",
            "description": "코드를 리뷰하고 버그를 탐지하는 검수자 에이전트. "
                           "버그 탐지, 보안 취약점, 성능 최적화, 가독성 평가를 담당합니다.",
            "model": "anthropic:claude-haiku-4-5-20251001",
            "system_prompt": """당신은 시니어 코드 리뷰어입니다.
다음 항목을 체계적으로 검토하세요:
1. 버그 및 논리 오류 (치명도: 높음/중간/낮음)
2. 보안 취약점 (SQL injection, XSS 등)
3. 성능 이슈 (시간복잡도, 메모리 사용)
4. 가독성 및 코드 스타일 (PEP 8)
5. 테스트 커버리지

각 항목을 1-10점으로 평가하고 개선 방안을 제시하세요.""",
            "tools": [],
        },
    ]
)

print("deepagents 3-에이전트 파이프라인 생성 완료")

## 섹션 7. 파이프라인 실행: 코딩 태스크 3개 테스트

피보나치, 퀵소트, API 클라이언트 3가지 태스크로 파이프라인을 테스트합니다.

In [ ]:
import time

# 테스트 태스크 정의
TEST_TASKS = [
    {
        "id": "fibonacci",
        "task": "피보나치 수열을 계산하는 Python 함수를 구현해주세요. "
                "재귀, 메모이제이션, 반복 세 가지 방법을 모두 구현하고 성능을 비교하세요.",
    },
    {
        "id": "quicksort",
        "task": "퀵소트 알고리즘을 Python으로 구현해주세요. "
                "피벗 선택 전략(첫 번째, 마지막, 중앙값)을 파라미터로 받을 수 있게 하세요.",
    },
    {
        "id": "api_client",
        "task": "REST API 클라이언트 클래스를 Python으로 구현해주세요. "
                "GET, POST, PUT, DELETE 메서드 지원, 재시도 로직, 타임아웃 처리를 포함하세요.",
    },
]

print(f"테스트 태스크 {len(TEST_TASKS)}개 준비 완료")

In [ ]:
# 3개 태스크 순차 실행
multi_agent_results = {}

for task_info in TEST_TASKS:
    task_id = task_info["id"]
    task_content = task_info["task"]
    
    print(f"\n{'='*60}")
    print(f"태스크: {task_id}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    # Langfuse 트레이싱 핸들러 생성
    handler = CallbackHandler(
        tags=["ep10", "deepagents", f"task-{task_id}"],
        session_id=f"ep10-deepagents-{task_id}",
    )
    
    # deepagents 실행
    result = code_review_agent.invoke(
        {"messages": [{"role": "user", "content": task_content}]},
        config={"callbacks": [handler]},
    )
    
    elapsed = time.time() - start_time
    
    # 결과 저장
    final_message = result["messages"][-1].content if result.get("messages") else ""
    multi_agent_results[task_id] = {
        "output": final_message,
        "elapsed_sec": round(elapsed, 2),
        "output_length": len(final_message),
    }
    
    print(f"완료 ({elapsed:.1f}초) | 출력: {len(final_message)} 글자")
    print(final_message[:300] + "..." if len(final_message) > 300 else final_message)

print("\n3개 태스크 모두 완료")

## 섹션 8. 단일 에이전트 vs 멀티 에이전트 품질 비교

LLM-as-Judge 방식으로 0~10점을 매겨 두 방식의 품질을 객관적으로 비교합니다.

In [ ]:
# 단일 에이전트로 동일 태스크 실행 (베이스라인)
single_agent_results = {}

for task_info in TEST_TASKS:
    task_id = task_info["id"]
    task_content = task_info["task"]
    
    start_time = time.time()
    
    response = llm.invoke([
        SystemMessage(content="당신은 Python 개발자입니다. 요청된 코드를 작성하고 간단한 리뷰도 제공하세요."),
        HumanMessage(content=task_content)
    ])
    
    elapsed = time.time() - start_time
    output = response.content
    
    single_agent_results[task_id] = {
        "output": output,
        "elapsed_sec": round(elapsed, 2),
        "output_length": len(output),
    }
    
    print(f"[단일] {task_id}: {elapsed:.1f}초 | {len(output)} 글자")

print("단일 에이전트 베이스라인 완료")

In [ ]:
# LLM-as-Judge: 0~10점 품질 평가
JUDGE_PROMPT = """당신은 소프트웨어 코드 품질 평가 전문가입니다.
다음 코드 작성 결과물을 0~10점으로 평가하세요.

평가 기준:
- 완성도: 요구사항이 모두 구현되었는가 (0~3점)
- 코드 품질: 타입힌트, docstring, 예외처리 등 (0~3점)
- 리뷰 깊이: 리뷰가 구체적이고 실용적인가 (0~4점)

반드시 다음 JSON 형식으로만 응답하세요:
{"score": <0~10 숫자>, "reason": "<간단한 이유>"}"""

def evaluate_output(task: str, output: str) -> dict:
    """LLM-as-Judge로 출력 품질 평가"""
    judge_llm = ChatAnthropic(
        model="claude-haiku-4-5-20251001",
        api_key=os.getenv("ANTHROPIC_API_KEY"),
    )
    
    response = judge_llm.invoke([
        SystemMessage(content=JUDGE_PROMPT),
        HumanMessage(content=f"태스크: {task}\n\n결과물:\n{output[:2000]}")
    ])
    
    try:
        evaluation = json.loads(response.content)
    except (json.JSONDecodeError, AttributeError):
        evaluation = {"score": 5.0, "reason": "파싱 실패"}
    
    return evaluation


# 모든 태스크에 대해 평가 실행
print("LLM-as-Judge 품질 평가 시작...\n")
comparison_results = []

for task_info in TEST_TASKS:
    task_id = task_info["id"]
    task_content = task_info["task"]
    
    # 단일 에이전트 평가
    single_eval = evaluate_output(task_content, single_agent_results[task_id]["output"])
    
    # 멀티 에이전트 평가
    multi_eval = evaluate_output(task_content, multi_agent_results[task_id]["output"])
    
    comparison_results.append({
        "task": task_id,
        "single_score": single_eval["score"],
        "multi_score": multi_eval["score"],
        "improvement": round(multi_eval["score"] - single_eval["score"], 1),
    })
    
    print(f"{task_id}: 단일={single_eval['score']:.1f} | 멀티={multi_eval['score']:.1f} | 향상={multi_eval['score'] - single_eval['score']:+.1f}")

# 평균 계산
avg_single = sum(r["single_score"] for r in comparison_results) / len(comparison_results)
avg_multi = sum(r["multi_score"] for r in comparison_results) / len(comparison_results)
avg_improvement = avg_multi - avg_single

print(f"\n평균: 단일={avg_single:.1f} | 멀티={avg_multi:.1f} | 향상={avg_improvement:+.1f}")

## 섹션 9. 처리 시간 및 비용 측정

In [ ]:
# 처리 시간 비교
print("=" * 60)
print(f"{'태스크':<15} {'단일(초)':<12} {'멀티(초)':<12} {'시간 차이':<12}")
print("-" * 60)

for task_info in TEST_TASKS:
    task_id = task_info["id"]
    single_time = single_agent_results[task_id]["elapsed_sec"]
    multi_time = multi_agent_results[task_id]["elapsed_sec"]
    diff = multi_time - single_time
    print(f"{task_id:<15} {single_time:<12.1f} {multi_time:<12.1f} {diff:+.1f}초")

print("=" * 60)
print("\n분석:")
print("- 멀티에이전트는 단일 에이전트보다 처리 시간이 더 길 수 있습니다")
print("- 이는 여러 LLM 호출이 발생하기 때문입니다 (기획 + 개발 + 검수)")
print("- 품질 향상 vs 시간/비용 트레이드오프를 고려해야 합니다")

In [ ]:
# 비용 분석 (claude-haiku 기준 대략적 추정)
# Input: $0.25/1M tokens, Output: $1.25/1M tokens
HAIKU_INPUT_COST = 0.25 / 1_000_000  # per token
HAIKU_OUTPUT_COST = 1.25 / 1_000_000  # per token

def estimate_tokens(text: str) -> int:
    """대략적인 토큰 수 추정 (1토큰 ≈ 4글자 영어, 2글자 한국어)"""
    return len(text) // 3

print("=== 비용 추정 (claude-haiku-4-5 기준) ===")
print()

for task_info in TEST_TASKS:
    task_id = task_info["id"]
    
    # 단일 에이전트: 1회 LLM 호출
    single_output_tokens = estimate_tokens(single_agent_results[task_id]["output"])
    single_input_tokens = estimate_tokens(task_info["task"]) + 100
    single_cost = (single_input_tokens * HAIKU_INPUT_COST + single_output_tokens * HAIKU_OUTPUT_COST) * 1000  # 원화 환산 대신 밀리달러
    
    # 멀티 에이전트: 약 4~6회 LLM 호출 추정
    multi_output_tokens = estimate_tokens(multi_agent_results[task_id]["output"])
    multi_input_tokens = estimate_tokens(task_info["task"]) * 3 + 500  # 3 에이전트 × 입력
    multi_cost = (multi_input_tokens * HAIKU_INPUT_COST + multi_output_tokens * HAIKU_OUTPUT_COST) * 1000
    
    print(f"{task_id}:")
    print(f"  단일 에이전트: ~${single_cost:.4f} (추정)")
    print(f"  멀티 에이전트: ~${multi_cost:.4f} (추정)")
    print(f"  비용 배수: {multi_cost/single_cost:.1f}x")
    print()

print("참고: Langfuse 대시보드에서 실제 토큰 사용량을 확인하세요")

## 섹션 10. Langfuse CallbackHandler 통합 + 트레이스 트리 확인법

멀티에이전트 실행 흐름을 Langfuse 대시보드에서 트리 구조로 시각화하는 방법을 안내합니다.

In [ ]:
# Langfuse 트레이싱이 적용된 deepagents 실행
trace_handler = CallbackHandler(
    tags=["ep10", "langfuse-demo", "trace-tree"],
    session_id="ep10-trace-demo",
    user_id="demo-user",
)

# 추적할 태스크 실행
print("Langfuse 트레이싱과 함께 파이프라인 실행 중...")
traced_result = code_review_agent.invoke(
    {"messages": [{"role": "user", "content": "버블 정렬 알고리즘을 Python으로 구현하고 리뷰해주세요"}]},
    config={"callbacks": [trace_handler]},
)

# 트레이스 ID 추출
trace_id = trace_handler.get_trace_id()
print(f"\nLangfuse 트레이스 ID: {trace_id}")
print(f"대시보드에서 확인: https://cloud.langfuse.com/traces/{trace_id}")

print("\n=== 트레이스 트리 구조 ===")
print("""
Trace (루트)
└── Span: Main Agent (기획자)
    ├── LLM: 요구사항 분석
    ├── Span: Subagent - developer
    │   └── LLM: 코드 작성
    ├── Span: Subagent - reviewer  
    │   └── LLM: 코드 리뷰
    └── LLM: 결과 통합
""")

In [ ]:
# Langfuse score API로 품질 점수 기록
if trace_id:
    # 멀티에이전트 결과 품질 평가
    traced_output = traced_result["messages"][-1].content if traced_result.get("messages") else ""
    eval_result = evaluate_output("버블 정렬 알고리즘 구현 + 리뷰", traced_output)
    
    # Langfuse에 점수 기록
    langfuse.score(
        trace_id=trace_id,
        name="overall_quality",
        value=eval_result["score"] / 10,  # 0~1 정규화
        comment=eval_result["reason"],
    )
    
    langfuse.score(
        trace_id=trace_id,
        name="multi_agent_pipeline",
        value=1.0,  # 멀티에이전트 파이프라인 사용 여부
        comment="3-에이전트 코드 리뷰 파이프라인 (기획자+개발자+검수자)",
    )
    
    print(f"Langfuse score 기록 완료")
    print(f"  overall_quality: {eval_result['score'] / 10:.2f}")
    print(f"  이유: {eval_result['reason']}")

## 섹션 11. 에이전트 간 통신 패턴 분석 (State 확인)

LangGraph 실행 후 State를 검사하여 에이전트 간 데이터 흐름을 분석합니다.

In [ ]:
# State 스트리밍으로 각 노드 실행 과정 추적
print("=== LangGraph State 흐름 추적 ===")
print()

stream_input = {
    "messages": [HumanMessage(content="팩토리얼 함수를 Python으로 구현해주세요")],
    "code": "",
    "review": "",
    "next": "",
}

step_count = 0
for chunk in supervisor_graph.stream(
    stream_input,
    config={"recursion_limit": 10},
    stream_mode="updates",
):
    step_count += 1
    for node_name, node_update in chunk.items():
        print(f"Step {step_count} | 노드: {node_name}")
        
        # 주요 State 필드 출력
        if "next" in node_update:
            print(f"  next: {node_update['next']}")
        if "code" in node_update and node_update["code"]:
            print(f"  code: {node_update['code'][:80]}...")
        if "review" in node_update and node_update["review"]:
            print(f"  review: {node_update['review'][:80]}...")
        if "messages" in node_update:
            msgs = node_update["messages"]
            print(f"  messages: {len(msgs)}개 추가")
        print()

print(f"총 {step_count}단계 실행 완료")

In [ ]:
# 최종 State 메시지 히스토리 분석
final_state = supervisor_graph.invoke(
    stream_input,
    config={"recursion_limit": 10}
)

print("=== 최종 메시지 히스토리 ===")
for i, msg in enumerate(final_state["messages"]):
    msg_type = type(msg).__name__
    content_preview = msg.content[:80].replace("\n", " ") if msg.content else ""
    print(f"[{i+1}] {msg_type}: {content_preview}...")

print(f"\n총 메시지 수: {len(final_state['messages'])}")

## 섹션 12. 무한 핑퐁 방지: recursion_limit + interrupt 가드레일

멀티에이전트 시스템에서 에이전트들이 서로를 무한 호출하는 상황을 방지합니다.

In [ ]:
from langgraph.errors import GraphRecursionError

# recursion_limit 실험: 매우 낮게 설정하여 강제 종료 확인
print("=== recursion_limit 가드레일 테스트 ===")
print()

low_limit_input = {
    "messages": [HumanMessage(content="복잡한 시스템 설계를 도와주세요")],
    "code": "",
    "review": "",
    "next": "",
}

try:
    # 매우 낮은 recursion_limit (2) → 강제 종료
    result = supervisor_graph.invoke(
        low_limit_input,
        config={"recursion_limit": 2},
    )
    print("정상 완료 (recursion_limit 내에 처리)")
except GraphRecursionError as e:
    print(f"GraphRecursionError 발생: {e}")
    print()
    print("→ 이것이 무한 핑퐁 방지 메커니즘입니다")
    print("→ 프로덕션에서는 적절한 recursion_limit을 설정하세요 (기본값: 25)")

In [ ]:
# 가드레일 패턴: 최대 위임 횟수 추적
class GuardedAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    code: str
    review: str
    delegation_count: int  # 위임 횟수 추적


MAX_DELEGATIONS = 3  # 최대 위임 횟수


def guarded_supervisor_node(state: GuardedAgentState) -> Command:
    """가드레일이 있는 Supervisor"""
    
    delegation_count = state.get("delegation_count", 0)
    
    # 최대 위임 횟수 초과 시 강제 종료
    if delegation_count >= MAX_DELEGATIONS:
        print(f"[Supervisor] 최대 위임 횟수 ({MAX_DELEGATIONS}회) 초과 → 강제 종료")
        return Command(goto="__end__", update={"next": "FORCE_FINISH"})
    
    # 정상 로직 (작업 완료 여부 체크)
    if not state.get("code"):
        next_agent = "code_writer"
    elif not state.get("review"):
        next_agent = "code_reviewer"
    else:
        next_agent = "__end__"
    
    print(f"[Supervisor] 위임 #{delegation_count + 1}: {next_agent}")
    
    return Command(
        goto=next_agent,
        update={
            "next": next_agent,
            "delegation_count": delegation_count + 1,
        }
    )


print("가드레일 패턴 구현 완료")
print(f"최대 위임 횟수: {MAX_DELEGATIONS}회")
print()
print("무한 핑퐁 방지 체크리스트:")
print("  1. recursion_limit 설정 (LangGraph 기본: 25)")
print("  2. delegation_count 추적으로 위임 횟수 제한")
print("  3. 각 에이전트의 종료 조건 명확히 정의")
print("  4. Langfuse로 비정상적으로 긴 트레이스 모니터링")

## 섹션 13. Exercise

아래 두 가지 Exercise를 완성하세요.

### Exercise 1: LangGraph Supervisor로 2-에이전트 파이프라인

**목표**: 섹션 4의 Supervisor 패턴을 참고하여, **새로운 2-에이전트 파이프라인**을 직접 구현하세요.

**요구사항**:
1. `AgentState`에 커스텀 필드 추가 (예: `summary`, `translated_text` 등)
2. Worker 1과 Worker 2의 역할을 자유롭게 정의 (번역+요약, 분석+보고서 작성 등)
3. `recursion_limit=8`로 3가지 입력을 테스트
4. `CallbackHandler`로 Langfuse 트레이싱 적용
5. Langfuse 대시보드에서 각 실행의 트레이스 트리 스크린샷 첨부

**힌트**: Supervisor의 `system_prompt`에 두 Worker의 역할을 명확히 설명하면 위임 결정이 더 정확해집니다.

In [ ]:
# Exercise 1 코드 작성 공간
# TODO: 2-에이전트 LangGraph Supervisor 파이프라인 구현

# 1. AgentState 정의
class MyAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str
    # TODO: 커스텀 필드 추가

# 2. Supervisor 노드 정의
def my_supervisor(state: MyAgentState):
    # TODO: 구현
    pass

# 3. Worker 1 노드 정의
def my_worker_1(state: MyAgentState):
    # TODO: 구현
    pass

# 4. Worker 2 노드 정의
def my_worker_2(state: MyAgentState):
    # TODO: 구현
    pass

# 5. StateGraph 구성 및 실행
# TODO: builder = StateGraph(MyAgentState)
# TODO: 노드 등록, 엣지 연결, 컴파일, 실행

### Exercise 2: deepagents 서브에이전트로 연구 → 작성 파이프라인

**목표**: `create_deep_agent`의 `subagents`를 활용하여 **연구자 → 작성자** 2단계 파이프라인을 구현하세요.

**요구사항**:
1. Main Agent: 주제 분석 + 파이프라인 조율 (기획자 역할)
2. Subagent 1 (`researcher`): 주어진 주제를 깊이 있게 분석하고 핵심 정보 요약
3. Subagent 2 (`writer`): 수집된 정보로 구조화된 기술 문서 작성 (목차 포함)
4. 3가지 주제로 테스트:
   - "Python async/await 동작 원리"
   - "LangGraph 핵심 개념과 사용 패턴"
   - "벡터 데이터베이스 종류별 비교"
5. 단일 에이전트 결과와 LLM-as-Judge(0~10점)로 품질 비교
6. Langfuse `score` API로 평가 결과 기록

**평가 기준**: 구조화 품질, 내용의 깊이, 가독성

In [ ]:
# Exercise 2 코드 작성 공간
# TODO: deepagents 연구 → 작성 파이프라인 구현

# 1. create_deep_agent로 파이프라인 구성
research_writing_agent = create_deep_agent(
    model="anthropic:claude-haiku-4-5-20251001",
    tools=[],
    system_prompt="""
    # TODO: Main Agent 시스템 프롬프트 작성
    """,
    subagents=[
        {
            "name": "researcher",
            "description": "TODO: researcher 설명",
            "model": "anthropic:claude-haiku-4-5-20251001",
            "system_prompt": "TODO: researcher 시스템 프롬프트",
            "tools": [],
        },
        {
            "name": "writer",
            "description": "TODO: writer 설명",
            "model": "anthropic:claude-haiku-4-5-20251001",
            "system_prompt": "TODO: writer 시스템 프롬프트",
            "tools": [],
        },
    ]
)

# 2. 3가지 주제로 테스트
RESEARCH_TOPICS = [
    "Python async/await 동작 원리",
    "LangGraph 핵심 개념과 사용 패턴",
    "벡터 데이터베이스 종류별 비교",
]

# TODO: 각 주제에 대해 단일/멀티 에이전트 실행 후 LLM-as-Judge 평가
# TODO: Langfuse score API로 평가 결과 기록
# TODO: 결과 비교표 출력